In [0]:
from pyspark.sql import SparkSession, DataFrame
from datetime import datetime
from pyspark.sql.functions import to_timestamp, col, date_trunc, current_timestamp, to_date, concat_ws, sha2
import logging

In [0]:
def date_now() -> datetime:
    return datetime.strptime(datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "%Y-%m-%d %H:%M:%S")

In [0]:
spark = SparkSession.builder \
    .appName("Ingestion from csv to delta") \
    .getOrCreate()

logging.basicConfig(
    level=logging.INFO
    , format="%(asctime)s | %(levelname)s | %(message)s"
    , datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

In [0]:
def read_stage(stage_name: str) -> DataFrame: 
    df_stage = spark.table(f"`comercial-{ambiente}`.stage.{stage_name}")
    return df_stage


In [0]:
df_stage = spark.table(f"`comercial-{ambiente}`.stage.{stage_name}")
display(df_stage)

In [0]:
dbutils.widgets.text("bronze_table_name", "", "Bronze name")
dbutils.widgets.text("stage_table_name", "", "Stage name")
dbutils.widgets.dropdown("ambiente", "dev", ["dev", "prod"])
dbutils.widgets.text("keys", "", "keys")

bronze_name = dbutils.widgets.get("bronze_table_name")
stage_name = dbutils.widgets.get("stage_table_name")
ambiente = dbutils.widgets.get("ambiente")
keys = dbutils.widgets.get("keys")

In [0]:
logs_stage = {
    "dat_Hr_Process_Start": date_now(),
    "dat_Hr_Process_End":  '1900-01-01 00:00:00',
    "layer": "bronze",
    "catalog": f'comercial-{ambiente}'.upper(),
    "source_Name": stage_name.upper(),
    "target_Name": bronze_name.upper(),
    "status": None,
    "count_Rows": None,
}

In [0]:
def bronze_exists(bronze_name: str) -> bool:
    return spark.catalog.tableExists(f'`comercial-{ambiente}`.`bronze`.`{bronze_name}`')

In [0]:
def merge_stage_bronze_table(stage: str, bronze_name: str):
    if bronze_exists(bronze_name):
        bronze_name.alias("source") \
            .merge(
                stage.alias("target"),
                "source.hash_column = target.hash_column"
            ) \
            .whenNotMatchedInsertAll()
    else: 
        stage.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f'`comercial-{ambiente}`.`bronze`.`{bronze_name}`')

def save_logs(logs):
    logs.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(f'`comercial-{ambiente}`.`corporate`.`logs`') 

In [0]:

try:
    logger.info("===========Start Process===========")
    logger.info("Start reading stage table")
    df_stage = read_stage(stage_name)
    logger.info("End reading stage table")


    logger.info("Start Ingestion Bronze")
    merge_stage_bronze_table(df_stage, bronze_name)
    logger.info("End Ingestion Bronze")
    

    logger.info("Start save logs")

    logs_stage["status"] = "SUCCESS"
    logs_stage["count_Rows"] = df_stage.count()
    logs_stage["dat_Hr_Process_End"] = date_now()

    df_logs = spark.createDataFrame([logs_stage])
    
    save_logs(df_logs)
    logger.info("End saving logs")
    logger.info("===========End Process===========")
except Exception as e:
    logger.error(f'{e}')

    
